# Model 4: Gradient Boosted Trees (OneVsRest)
## IoT Saldırı Tipi Sınıflandırması (Multi-class)

Bu notebook, **Edge-IIoTset** veri seti üzerinde **Attack_type** kolonunu hedef değişken olarak kullanan
Gradient Boosted Trees (GBT) modelini **OneVsRest** wrapper ile adım adım eğitir ve değerlendirir.

### Neden GBT?
- **Boosting**: Zayıf öğrenicileri sıralı olarak güçlendirerek yüksek doğruluk elde eder
- **Gradient tabanlı optimizasyon**: Her yeni ağaç önceki hataları düzeltir
- **Yüksek performans**: Çoğu tabular veri yarışmasında en iyi sonuçlardan birini verir
- **Feature importance**: Ağaçların birleşik katkısı ile güvenilir önem skoru

### OneVsRest Neden Gerekli?
Spark MLlib'in `GBTClassifier`'ı **doğrudan multi-class desteklemez**. Bu yüzden `OneVsRest` wrapper kullanılır:
- N sınıf → N adet binary GBT eğitilir
- Her sınıf için "bu sınıf mı / değil mi" şeklinde binary sınıflandırma yapılır
- En yüksek skoru veren sınıf tahmin olarak seçilir
- **Dezavantaj**: N kat daha yavaş eğitim

### Pipeline Adımları
1. Gold Delta Lake → Veri yükleme
2. Feature vektörleme + StringIndexer
3. Sınıf ağırlıkları (classWeight)
4. Stratified Train/Test split (%80/%20)
5. GBTClassifier + OneVsRest
6. CrossValidator ile hiperparametre arama (maxIter, maxDepth)
7. Test seti değerlendirme
8. Feature importance (sınıflar arası ortalama)
9. MLflow loglama

## 1. Kütüphaneler ve Spark Session

In [ ]:
import sys
import time
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from pyspark.ml import Pipeline
from pyspark.ml.classification import GBTClassifier, OneVsRest
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

sys.path.insert(0, "/opt/bitnami/spark")

from spark.spark_session import get_spark
from ml.utils import (
    run_ml_pipeline_multiclass,
    evaluate_model_multiclass,
    compute_confusion_matrix_multiclass,
    log_to_mlflow,
)

print("Kütüphaneler yüklendi.")

In [ ]:
spark = get_spark("Notebook-GBT-OneVsRest-Multiclass")
print(f"Spark version: {spark.version}")
print(f"App name: {spark.sparkContext.appName}")

## 2. Veri Yükleme ve Feature Hazırlığı

In [ ]:
SAMPLE_SIZE = 30000  # Hızlı test için (None = tüm veri)

stage_start = time.perf_counter()
train_df, test_df, feature_cols, label_index_model = run_ml_pipeline_multiclass(
    spark,
    sample_size=SAMPLE_SIZE,
    split_log_stats=True,
)
label_names = list(label_index_model.labels)
num_classes = len(label_names)

print(f"\nPipeline süresi: {time.perf_counter() - stage_start:.2f}s")
print(f"Feature sayısı: {len(feature_cols)}")
print(f"Sınıf sayısı: {num_classes}")
print(f"Sınıflar: {label_names}")
print(f"\nOneVsRest ile {num_classes} adet binary GBT eğitilecek.")

In [ ]:
train_count = train_df.count()
test_count = test_df.count()
print(f"Train: {train_count:,} satır")
print(f"Test:  {test_count:,} satır")

print("\nTrain seti sınıf dağılımı:")
train_df.groupBy("label").count().orderBy("label").show()

## 3. Model Pipeline Oluşturma

GBT + OneVsRest pipeline'ı:
- **GBTClassifier**: Base binary sınıflandırıcı (her sınıf için birer tane)
- **OneVsRest**: N sınıf için N tane binary GBT eğitir

### Hiperparametreler
| Parametre | Açıklama | Aranacak Değerler |
|-----------|----------|------------------|
| `maxIter` | Boosting iterasyon (ağaç) sayısı | 20, 50 |
| `maxDepth` | Her ağacın maksimum derinliği | 3, 5 |
| `stepSize` | Learning rate | 0.1 (sabit) |

**Not**: OneVsRest çok pahalı olduğu için grid sade tutulmuştur.

In [ ]:
gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight",
    maxIter=30,
    maxDepth=5,
    stepSize=0.1,
    seed=42,
)

ovr = OneVsRest(
    classifier=gbt,
    labelCol="label",
    featuresCol="features",
    weightCol="classWeight",
    parallelism=1,  # Binary GBT'ler sıralı eğitilsin (RAM dostu)
)

pipeline = Pipeline(stages=[ovr])
print("Pipeline oluşturuldu: GBTClassifier + OneVsRest")
print(f"Her fold'da {num_classes} binary GBT eğitilecek.")

## 4. Cross Validation

In [ ]:
max_iter_values = [20, 50]
max_depth_values = [3, 5]
num_folds = 2

param_grid = (
    ParamGridBuilder()
    .addGrid(gbt.maxIter, max_iter_values)
    .addGrid(gbt.maxDepth, max_depth_values)
    .build()
)

cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="f1",
    ),
    numFolds=num_folds,
    seed=42,
    parallelism=2,
)

total_cv_runs = len(param_grid) * num_folds * num_classes
print(f"Grid boyutu: {len(param_grid)} kombinasyon")
print(f"Fold sayısı: {num_folds}")
print(f"OneVsRest çarpanı: {num_classes} sınıf")
print(f"Toplam binary GBT fit: {total_cv_runs}")

## 5. Model Eğitimi

**Uyarı**: OneVsRest + GBT eğitimi diğer modellere kıyasla çok daha uzun sürer.

In [ ]:
print("Cross Validation başlatılıyor (OneVsRest — uzun sürebilir)...")
cv_start = time.perf_counter()
cv_model = cv.fit(train_df)
cv_duration = time.perf_counter() - cv_start
print(f"CV tamamlandı! Süre: {cv_duration:.2f}s")

# En iyi model parametreleri
best_ovr_model = cv_model.bestModel.stages[-1]
first_sub = best_ovr_model.models[0]  # İlk binary GBT'den parametre oku
best_max_iter = int(first_sub.getOrDefault("maxIter"))
best_max_depth = int(first_sub.getOrDefault("maxDepth"))
best_step_size = float(first_sub.getOrDefault("stepSize"))

print(f"\nEn iyi parametreler:")
print(f"  maxIter:  {best_max_iter}")
print(f"  maxDepth: {best_max_depth}")
print(f"  stepSize: {best_step_size}")

In [ ]:
# CV sonuçları
avg_metrics = cv_model.avgMetrics
print("CV Sonuçları (F1-Score):")
print(f"{'#':<4} {'Parametreler':<35} {'Avg F1':>10}")
print("-" * 52)
cv_details = []
for i, (params, score) in enumerate(zip(param_grid, avg_metrics)):
    ps = ", ".join(f"{p.name}={v}" for p, v in params.items())
    marker = " ← best" if i == max(range(len(avg_metrics)), key=lambda x: avg_metrics[x]) else ""
    print(f"{i+1:<4} {ps:<35} {score:>10.4f}{marker}")
    cv_details.append({"index": i, "score": float(score), "params": ps})

cv_best_f1 = float(max(avg_metrics)) if avg_metrics else 0.0

## 6. Test Seti Değerlendirme

In [ ]:
predictions = cv_model.bestModel.transform(test_df)
metrics = evaluate_model_multiclass(predictions, num_classes=num_classes)

In [ ]:
confusion = compute_confusion_matrix_multiclass(predictions, label_names=label_names)

In [ ]:
print("\n" + "=" * 40)
print("SONUÇ ÖZETİ")
print("=" * 40)
print(f"Accuracy:  {metrics.get('accuracy', 0):.4f}")
print(f"F1-Score:  {metrics.get('f1_score', 0):.4f}")
print(f"Precision: {metrics.get('precision', 0):.4f}")
print(f"Recall:    {metrics.get('recall', 0):.4f}")

## 7. Feature Importance

OneVsRest'te her sınıf için ayrı bir binary GBT eğitilir. Feature importance hesaplamak için
tüm binary modellerin `featureImportances` değerlerinin **aritmetik ortalaması** alınır.

In [ ]:
# OneVsRest feature importance: tüm binary GBT'lerin ortalaması
sub_models = best_ovr_model.models
n_features = len(feature_cols)
agg = [0.0] * n_features

for sm in sub_models:
    imp = sm.featureImportances.toArray().tolist()
    for j in range(n_features):
        agg[j] += imp[j] if j < len(imp) else 0.0

agg = [v / max(len(sub_models), 1) for v in agg]
fi_pairs = sorted(zip(feature_cols, agg), key=lambda x: x[1], reverse=True)
top_features = fi_pairs[:15]

print("Top 15 Feature Importance (sınıflar arası ortalama):")
print(f"{'#':<4} {'Feature':<35} {'Importance':>12}")
print("-" * 55)
for idx, (fname, importance) in enumerate(top_features, start=1):
    print(f"{idx:<4} {fname:<35} {importance:>12.6f}")

In [ ]:
# Feature Importance Görselleştirmesi
top10 = fi_pairs[:10]
names = [f[0] for f in reversed(top10)]
values = [f[1] for f in reversed(top10)]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(names, values, color="#FF9800", edgecolor="#E65100", height=0.6)
for bar, val in zip(bars, values):
    ax.text(bar.get_width() + max(values) * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}", va="center", fontsize=9, fontweight="bold")

ax.set_xlabel("Mean Feature Importance (OneVsRest GBT)", fontsize=11)
ax.set_title("GBT — Top 10 Feature Importance (Multi-class)",
             fontsize=13, fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

## 8. Sınıf Bazında Binary GBT Analizi

In [ ]:
# Her binary GBT modeli hakkında bilgi
print(f"OneVsRest: {len(sub_models)} binary GBT modeli\n")
print(f"{'Sınıf':<25} {'Ağaç Sayısı':>12} {'Toplam Düğüm':>14}")
print("-" * 55)
for i, sm in enumerate(sub_models):
    cls_name = label_names[i] if i < len(label_names) else f"class_{i}"
    n_trees = sm.getNumTrees
    total_nodes = sm.totalNumNodes
    print(f"{cls_name:<25} {n_trees:>12} {total_nodes:>14}")

## 9. MLflow'a Loglama

In [ ]:
best_params = {
    "maxIter": best_max_iter,
    "maxDepth": best_max_depth,
    "stepSize": best_step_size,
    "numClasses": int(num_classes),
    "numFolds": num_folds,
    "grid_size": len(param_grid),
    "cv_total_binary_fits": total_cv_runs,
    "wrapper": "OneVsRest",
    "weightCol": "classWeight",
    "label_column": "Attack_type",
}

metrics["cv_best_f1"] = cv_best_f1

confusion_metrics = {}
for i, name in enumerate(label_names):
    confusion_metrics[f"row_total_class_{i}"] = confusion["row_totals"][i]
    confusion_metrics[f"per_class_acc_{i}"] = confusion["per_class_acc"][i]

run_id = log_to_mlflow(
    run_name="gbt_notebook",
    model_type="GBTClassifier",
    params=best_params,
    metrics={**metrics, **confusion_metrics},
    model=cv_model.bestModel,
    feature_importance=top_features[:10],
    tags={
        "source": "notebook",
        "model_index": "4",
        "classification_type": "multiclass",
        "ensemble_method": "boosting",
        "multiclass_wrapper": "OneVsRest",
    },
)
print(f"\nMLflow Run ID: {run_id}")

## Özet

Bu notebook'ta **Gradient Boosted Trees (OneVsRest)** modelini başarıyla eğittik:

- **Boosting** yöntemi ile sıralı olarak hataları düzelten ağaçlar eğitildi
- **OneVsRest** wrapper ile multi-class problemi N binary probleme dönüştürüldü
- **CrossValidator** ile maxIter ve maxDepth optimize edildi
- **classWeight** ile sınıf dengesizliği telafi edildi
- Binary GBT'lerin birleşik **feature importance** analizi yapıldı
- Sonuçlar **MLflow**'a loglandı

GBT, genellikle en yüksek doğruluğu veren model olmakla birlikte, OneVsRest nedeniyle eğitim süresi uzundur.

In [ ]:
spark.stop()
print("Spark oturumu kapatıldı.")